# Portable scientific figures in notebooks
Build packages, run `node python/build-notebook.mjs`, and install the private checkout with `python -m pip install -e "./python[notebook,numpy,arrow]"`. No registry publication is needed. Run this notebook from the repository root (or set MARGO_REPO).

The same anywidget displays in JupyterLab, VS Code and Colab. In marimo wrap the widget with `mo.ui.anywidget(exo)`. Observe `camera` and `selection` through traitlets; assign them to drive all open views. `exo.save_html("exoplanets.html")` writes offline HTML for Quarto or an Observable iframe. Full host and static export guidance: docs/guides/sciviz-notebooks.md.


In [ ]:
import gzip, json, os
from pathlib import Path
import numpy as np
import pyarrow as pa
from margoengine.notebook import NotebookFigure
root = Path(os.environ.get("MARGO_REPO", Path.cwd()))


In [ ]:
# OpenNeuro ds000030 sub-10159 T1w, CC0. Existing pinned reduction: head.json.
header, compressed = (root / "examples/assets/sciviz/head.nrrd").read_bytes().split(b"\n\n", 1)
values = np.frombuffer(gzip.decompress(compressed), dtype="<u2")
bounds = [0, 174, 0, 254, 0, 254]
mri = NotebookFigure(options={"axes": False, "aspect": [174 / 254, 1, 1]}, limits=bounds, height=360)
mri.add("volume", values, {"dims": [88, 128, 128], "bounds": bounds, "range": [0, 600], "colormap": "gray", "density": 2})
mri


In [ ]:
# NASA Exoplanet Archive, public data. Pinned source and licence: stats/source.json.
rows = json.loads((root / "examples/assets/sciviz/stats/exoplanets.json").read_text())
table = pa.Table.from_pylist(rows)
x, y = table["pl_orbper"], table["pl_rade"]
exo = NotebookFigure(options={"axes": False, "titles": {"x": "Orbital period (days)", "y": "Radius (Earth radii)"}},
                     limits=[0, 70, 0, 20, -1, 1], height=360)
exo.add("scatter", {"x": x, "y": y, "z": np.zeros(len(rows), dtype="<f4"), "color": y}, {"size": 7})
exo
